In [1]:
import pandas as pd

concept_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_concepts_exports"
icustay_detail_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports/icu_icustays.csv"

vaso_files = {
    "norepinephrine": "norepinephrine.csv",
    "epinephrine": "epinephrine.csv",
    "dopamine": "dopamine.csv",
    "dobutamine": "dobutamine.csv",
    "phenylephrine": "phenylephrine.csv",
    "vasopressin": "vasopressin.csv"
}

icustays = pd.read_csv(icustay_detail_path, dtype={"subject_id": str, "hadm_id": str, "stay_id": str})

all_vaso = []

for drug, fname in vaso_files.items():
    df = pd.read_csv(f"{concept_path}/{fname}", dtype={"stay_id": str})
    
    df_proc = df.rename(columns={
        "stay_id": "stay_id",
        "starttime": "med_start",
        "endtime": "med_stop",
        "vaso_amount": "med_action_dose"
    })
    
    df_proc["drug"] = drug
    df_proc["med_action_dose_unit"] = "mcg/kg/min"  
    df_proc = df_proc[["stay_id", "med_start", "med_stop", "med_action_dose", "med_action_dose_unit", "drug"]]
    
    all_vaso.append(df_proc)

vaso_df = pd.concat(all_vaso, ignore_index=True)

vaso_df = vaso_df.merge(
    icustays[["subject_id", "hadm_id", "stay_id"]],
    on="stay_id",
    how="left"
)

vaso_df["csn"] = vaso_df["hadm_id"]
vaso_df["pat_id"] = vaso_df["subject_id"]

df_vasopressor_meds = vaso_df[[
    "csn", "pat_id", "stay_id",
    "drug", "med_start", "med_stop", "med_action_dose", "med_action_dose_unit"
]]

print(df_vasopressor_meds.head())

out_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/INFUSIONMEDS/df_vasopressor_meds.csv"
df_vasopressor_meds.to_csv(out_path, index=False)
print(f"✅ Saved vasopressor meds to {out_path}")

        csn    pat_id   stay_id            drug            med_start  \
0  27793700  10002114  34672098  norepinephrine  2162-02-18 07:58:00   
1  27793700  10002114  34672098  norepinephrine  2162-02-18 08:38:00   
2  27793700  10002114  34672098  norepinephrine  2162-02-18 08:55:00   
3  27793700  10002114  34672098  norepinephrine  2162-02-18 09:12:00   
4  27793700  10002114  34672098  norepinephrine  2162-02-18 10:28:00   

              med_stop  med_action_dose med_action_dose_unit  
0  2162-02-18 08:38:00         0.128233           mcg/kg/min  
1  2162-02-18 08:55:00         0.032696           mcg/kg/min  
2  2162-02-18 09:12:00         0.054493           mcg/kg/min  
3  2162-02-18 10:28:00         6.042001           mcg/kg/min  
4  2162-02-18 10:42:00         0.026923           mcg/kg/min  
✅ Saved vasopressor meds to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/INFUSIONMEDS/df_vasopressor_meds.csv
